<a href="https://colab.research.google.com/github/saraxfl/Notebook/blob/main/collab_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Paquetes que vamos a usar
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import zipfile
import io

In [ ]:
print(" ---- FASE 1 : EXTRACIÓN ----- ")
# Descargar los viajes del mes
# Datos obtenidos del portal de CDMX
# Periodo: julio 2026
url = "https://ecobici.cdmx.gob.mx/wp-content/uploads/2026/08/public_data_web_2026-07.csv"
csv_file_name = "2026-07.csv"
print(fe"Descargando datos desde {url}")
try
  response =requests.get(url,timeout=1200) # tiempo amplio para la descarga
  response.raise_for_status() # validar respuesta del servidor
  print("Descarga con éxito")
except requests.exceptions.Timeout as e:
  print(f"Error de tiempo de espera: {e}")
  df_raw = pd.DataFrame()
except requests.exceptions.RequestException as e:
  print(f"Error durante descarga : {e}")
  df_raw = pd-pd.DataFrame()

 ---- FASE 1 : EXTRACIÓN ----- 
Descargando datos desde https://ecobici.cdmx.gob.mx/wp-content/uploads/2026/08/public_data_web_2026-07.csv


In [ ]:
# Guardar el archivo descargado
with open(csv_file_name,'wb') as f:
  f.write(response.content)
  print(f"Archivo CSV guardado como {csv_file_name}")

# Abrir el CSV en un DataFrame
print(f"Leyendo datos desde {csv_file_name}")
df_raw = pd.read_csv(csv_file_name) # cargar datos originales
print("Extracción completada con exito")
print(f"Se cargaron {df_raw.shape[0]} registros.")

Archivo CSV guardado como 2026-07.csv
Leyendo datos desde 2026-07.csv
Extracción completada con exito
Se cargaron 1493484 registros.


In [ ]:
# Revisar dimensiones del DataFrame
print("Tamaño del DataFrame: ")
print(df_raw.shape)
# Ver una muestra de los registros
print("Previsualización del DataFrame: ")
display(df_raw.head(1000))

Tamaño del DataFrame: 
(1493484, 9)
Previsualización del DataFrame: 


,Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_EstacionArribo,Fecha_Arribo,Hora_Arribo
0,F,26.0,5552989,085,30/06/2026,23:43:41,503,01/07/2026,00:00:00
1,M,33.0,5128335,259,30/06/2026,23:52:28,266-267,01/07/2026,00:00:00
2,M,34.0,8647703,040,30/06/2026,23:41:57,011,01/07/2026,00:00:03
3,M,34.0,5633250,492,30/06/2026,23:56:52,489,01/07/2026,00:00:03
4,O,41.0,8516015,133,30/06/2026,23:31:01,345,01/07/2026,00:00:05
...,...,...,...,...,...,...,...,...,...
995,M,36.0,8838658,331,01/07/2026,00:07:07,044,01/07/2026,00:28:40
996,M,25.0,4351136,232,01/07/2026,00:01:42,140,01/07/2026,00:28:41
997,F,32.0,2252880,120,01/07/2026,00:15:04,149,01/07/2026,00:28:42
998,M,28.0,3414604,260,30/06/2026,23:42:25,614,01/07/2026,00:28:44


In [ ]:
# FASE 2: TRANSFORMACIÓN
df = df_raw.copy()
# Pasar las fechas a formato datetime
df['Fecha_Retiro'] = pd.to_datetime(df['Fecha_Retiro'], dayfirst=True)
df['Fecha_Arribo'] = pd.to_datetime(df['Fecha_Arribo'], dayfirst=True)
print("Columnas de fecha convertidas a datetime.")
display(df.head())

Columnas de fecha convertidas a datetime.


,Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_EstacionArribo,Fecha_Arribo,Hora_Arribo
0,F,26.0,5552989,085,2026-06-30,23:43:41,503,2026-07-01,00:00:00
1,M,33.0,5128335,259,2026-06-30,23:52:28,266-267,2026-07-01,00:00:00
2,M,34.0,8647703,040,2026-06-30,23:41:57,011,2026-07-01,00:00:03
3,M,34.0,5633250,492,2026-06-30,23:56:52,489,2026-07-01,00:00:03
4,O,41.0,8516015,133,2026-06-30,23:31:01,345,2026-07-01,00:00:05


In [ ]:
# Crear variables nuevas
print("Iniciando Feature Engineerning ...")

# Unir fecha y hora para medir cada recorrido
df['Fecha_Retiro_Completa'] = pd.to_datetime(df['Fecha_Retiro'].dt.strftime('%Y-%m-%d') + ' ' + df['Hora_Retiro'])
df['Fecha_Arribo_Completa'] = pd.to_datetime(df['Fecha_Arribo'].dt.strftime('%Y-%m-%d') + ' ' + df['Hora_Arribo'])
# Generar columnas útiles para el análisis
# Duración del recorrido
df['duracion_minutos'] = (df['Fecha_Arribo_Completa'] - df['Fecha_Retiro_Completa']).dt.total_seconds() / 60
# Número del día de la semana
df['dia_semana'] = df['Fecha_Retiro'].dt.dayofweek
# Hora en la que inicia el viaje
df['hora_inicio'] = df['Fecha_Retiro_Completa'].dt.hour
# Separar entre semana y fin de semana
df['tipo_día'] = df['dia_semana'].apply(lambda x: 'Fin de semana' if x >= 5 else 'Entre semana')

print("Nuevas características creadas: 'duracion_minutos', 'dia_semana', 'hora_inicio', 'tipo_día'")
display(df.head())

Iniciando Feature Engineerning ...
Nuevas características creadas: 'duracion_minutos', 'dia_semana', 'hora_inicio', 'tipo_día'


,Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_EstacionArribo,Fecha_Arribo,Hora_Arribo,Fecha_Retiro_Completa,Fecha_Arribo_Completa,duracion_minutos,dia_semana,hora_inicio,tipo_día
0,F,26.0,5552989,085,2026-06-30,23:43:41,503,2026-07-01,00:00:00,2026-06-30 23:43:41,2026-07-01 00:00:00,16.316667,1,23,Entre semana
1,M,33.0,5128335,259,2026-06-30,23:52:28,266-267,2026-07-01,00:00:00,2026-06-30 23:52:28,2026-07-01 00:00:00,7.533333,1,23,Entre semana
2,M,34.0,8647703,040,2026-06-30,23:41:57,011,2026-07-01,00:00:03,2026-06-30 23:41:57,2026-07-01 00:00:03,18.100000,1,23,Entre semana
3,M,34.0,5633250,492,2026-06-30,23:56:52,489,2026-07-01,00:00:03,2026-06-30 23:56:52,2026-07-01 00:00:03,3.183333,1,23,Entre semana
4,O,41.0,8516015,133,2026-06-30,23:31:01,345,2026-07-01,00:00:05,2026-06-30 23:31:01,2026-07-01 00:00:05,29.066667,1,23,Entre semana


In [ ]:
# 2. 3 Normalización / Estandarización
# Vamos a normalizar la duración del viaje para que este en escala de 0 a 1
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df["'duracion_minutos' normalizamos a una escala de 0 a 1"] = scaler.fit_transform(df[['duracion_minutos']])
print("'duracion_minutos' normalizada a una escala de 0 a 1")
# 2.4 Encoding de variables génericas
# La variable 'tipo_dia' es catégorica. La convertiremos a números usando One-Hot Encoding
df_encoded = pd.get_dummies(df, columns=['tipo_día'], prefix='dia')
print("Variable 'tipo_día' codificada con One-Hot Encoding")
#2.5 Balanceo de clases
#Imaginemos que queremos predecir si un viaje es "muy largo" (>60)
df['viaje largo'] = df['duracion_minutos'] > 60
print("Ejemplo de Balanceo de clase:")
print(df['viaje largo'].value_counts())
# Verificación de DataFrame transformado
print("Vista previa del DataFrame transformado")
print(df_encoded.head())

'duracion_minutos' normalizada a una escala de 0 a 1
Variable 'tipo_día' codificada con One-Hot Encoding
Ejemplo de Balanceo de clase:
viaje largo
False    1484173
True        9311
Name: count, dtype: int64
Vista previa del DataFrame transformado
  Genero_Usuario  Edad_Usuario     Bici Ciclo_Estacion_Retiro Fecha_Retiro  \
0              F          26.0  5552989                   085   2026-06-30   
1              M          33.0  5128335                   259   2026-06-30   
2              M          34.0  8647703                   040   2026-06-30   
3              M          34.0  5633250                   492   2026-06-30   
4              O          41.0  8516015                   133   2026-06-30   

  Hora_Retiro Ciclo_EstacionArribo Fecha_Arribo Hora_Arribo  \
0    23:43:41                  503   2026-07-01    00:00:00   
1    23:52:28              266-267   2026-07-01    00:00:00   
2    23:41:57                  011   2026-07-01    00:00:03   
3    23:56:52                  4